# Calculadora de Juros Compostos
## Simulador interativo de investimentos


In [ ]:
import ipywidgets as widgets
from ipywidgets import Layout
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np


: 

In [ ]:
def simular(taxa_mensal, alvo, fases, mes_inicial=0, saldo_inicial=0,
            estrategia='aporte_depois'):
    saldo = float(saldo_inicial)
    total_investido = float(saldo_inicial)
    aportes_acum = float(saldo_inicial)
    meses_total = 0
    registros = []
    marcos = {}

    def processar_mes(aporte_valor, nome):
        nonlocal saldo, total_investido, aportes_acum, meses_total
        saldo_anterior = saldo
        if estrategia == 'aporte_depois':
            rendimento = saldo * taxa_mensal
            saldo = saldo * (1 + taxa_mensal) + aporte_valor
        else:
            saldo = (saldo + aporte_valor) * (1 + taxa_mensal)
            rendimento = saldo - saldo_anterior - aporte_valor
        total_investido += aporte_valor
        aportes_acum += aporte_valor
        meses_total += 1

        for pct in [0.25, 0.50, 0.75, 1.00]:
            if pct not in marcos and saldo >= alvo * pct:
                marcos[pct] = {
                    'mes': meses_total,
                    'ano': round(meses_total / 12, 2),
                    'saldo': round(saldo, 2),
                    'pct': pct * 100
                }

        registros.append({
            'Mes': meses_total,
            'Ano': round(meses_total / 12, 2),
            'Fase': nome,
            'Aporte': aporte_valor,
            'Rendimento': round(rendimento, 2),
            'Saldo': round(saldo, 2),
            'Aportes_Acum': round(aportes_acum, 2),
            '%Meta': round(saldo / alvo * 100, 2)
        })
        return saldo

    for meses_qtd, aporte_valor, nome in fases:
        if meses_qtd is None:
            while saldo < alvo:
                processar_mes(aporte_valor, nome)
        else:
            for _ in range(int(meses_qtd)):
                processar_mes(aporte_valor, nome)

    df = pd.DataFrame(registros)
    return {
        'saldo_final': round(saldo, 2),
        'total_investido': round(total_investido, 2),
        'total_juros': round(saldo - total_investido, 2),
        'meses_total': meses_total,
        'anos': meses_total // 12,
        'meses_resto': meses_total % 12,
        'df': df,
        'registros': registros,
        'marcos': marcos
    }


def get_fase(registro, mes):
    if mes <= 0:
        return 'Saldo atual'
    df = pd.DataFrame(registro)
    if mes > len(df):
        return 'Final'
    return df.iloc[mes - 1]['Fase']


def continuar_de(taxa_mensal, alvo, fases, mes_atual, saldo_atual,
                 estrategia='aporte_depois'):
    meses_acum = 0
    fases_restantes = []
    for meses_qtd, aporte_valor, nome in fases:
        if meses_qtd is None:
            if meses_acum <= mes_atual:
                fases_restantes.append((None, aporte_valor, nome))
            break
        inicio_fase = meses_acum
        fim_fase = meses_acum + meses_qtd
        if fim_fase > mes_atual:
            restantes = fim_fase - mes_atual
            if restantes > 0:
                fases_restantes.append((restantes, aporte_valor, nome))
        meses_acum += meses_qtd

    if not fases_restantes:
        aporte = fases[-1][1]
        fases_restantes = [(None, aporte, fases[-1][2])]

    return simular(taxa_mensal, alvo, fases_restantes,
                   mes_inicial=mes_atual, saldo_inicial=saldo_atual,
                   estrategia=estrategia)


In [ ]:
taxa_slider = widgets.FloatSlider(
    value=0.83, min=0.10, max=2.50, step=0.01,
    description='Taxa mensal:',
    style={'description_width': '120px'}, layout=Layout(width='400px')
)
taxa_label = widgets.Label('0.83% (10.44% a.a.)')

alvo_input = widgets.FloatText(
    value=1_000_000, min=1_000, max=100_000_000,
    description='Meta (R$):',
    style={'description_width': '120px'}, layout=Layout(width='300px')
)

ap1 = widgets.IntSlider(value=300, min=0, max=10000, step=50,
                        description='Fase 1:', style={'description_width': '120px'})
ap2 = widgets.IntSlider(value=500, min=0, max=10000, step=50,
                        description='Fase 2:', style={'description_width': '120px'})
ap3 = widgets.IntSlider(value=700, min=0, max=10000, step=50,
                        description='Fase 3:', style={'description_width': '120px'})
ap4 = widgets.IntSlider(value=1000, min=0, max=10000, step=50,
                        description='Fase 4:', style={'description_width': '120px'})

m1 = widgets.IntSlider(value=24, min=3, max=120, step=3,
                        description='Meses F1:', style={'description_width': '120px'})
m2 = widgets.IntSlider(value=24, min=3, max=120, step=3,
                        description='Meses F2:', style={'description_width': '120px'})
m3 = widgets.IntSlider(value=24, min=3, max=120, step=3,
                        description='Meses F3:', style={'description_width': '120px'})

mes_ini = widgets.IntSlider(value=0, min=0, max=400, step=1,
                             description='Mes inicial:', style={'description_width': '120px'})
saldo_ini = widgets.FloatText(value=0, min=0, max=1_000_000,
                               description='Saldo inicial:', style={'description_width': '120px'})

estrategia_dd = widgets.Dropdown(
    options=[('Juros primeiro, depois aporte', 'aporte_depois'),
             ('Aporte primeiro, depois juros', 'aporte_antes')],
    value='aporte_depois',
    description='Estrategia:',
    style={'description_width': '120px'}, layout=Layout(width='400px')
)

def fmt_taxa(change):
    pct = change['new']
    aa = ((1 + pct / 100) ** 12 - 1) * 100
    taxa_label.value = f'{pct:.2f}% ({aa:.2f}% a.a.)'

taxa_slider.observe(fmt_taxa, names='value')
fmt_taxa({'new': taxa_slider.value})

def on_mes_ini_change(change):
    mi = change['new']
    if mi > 0 and saldo_ini.value == 0:
        saldo_ini.value = 1000

mes_ini.observe(on_mes_ini_change, names='value')

aporte_box = widgets.VBox([
    widgets.HBox([ap1, ap2]),
    widgets.HBox([ap3, ap4]),
])
fases_box = widgets.VBox([
    widgets.HBox([m1, m2]),
    widgets.HBox([m3]),
])
config_box = widgets.VBox([
    widgets.HBox([taxa_slider, taxa_label]),
    alvo_input,
    widgets.HBox([mes_ini, saldo_ini]),
    widgets.HBox([estrategia_dd]),
])


In [ ]:
def atualizar(taxa_pct, alvo, a1, a2, a3, a4,
               m1_m, m2_m, m3_m, mi, si, est):
    clear_output(wait=True)

    if taxa_pct <= 0 or alvo <= 0:
        display(widgets.HTML('<h3 style="color:red">Preencha valores validos.</h3>'))
        return

    taxa = taxa_pct / 100
    fases = [
        (m1_m, a1, f'R$ {a1}/mes'),
        (m2_m, a2, f'R$ {a2}/mes'),
        (m3_m, a3, f'R$ {a3}/mes'),
        (None, a4, f'R$ {a4}/mes'),
    ]

    if mi > 0 and si > 0:
        res = continuar_de(taxa, alvo, fases, mi, si, est)
    else:
        res = simular(taxa, alvo, fases, estrategia=est)

    df = res['df']
    marcos = res['marcos']
    anos = res['anos']
    meses_resto = res['meses_resto']
    sf = res['saldo_final']
    ti = res['total_investido']
    tj = res['total_juros']

    if len(df) == 0:
        display(widgets.HTML('<h3 style="color:red">Nao foi possivel atingir a meta.</h3>'))
        return

    # ======== RESUMO EXECUTIVO ========
    display(widgets.HTML(f'''
    <div style="background:#f0f8ff;padding:15px;border-radius:8px;border:1px solid #bcd;
                font-family:monospace;margin-bottom:20px">
      <h3 style="margin:0 0 10px 0;color:#1565C0">Resumo Executivo</h3>
      <table>
        <tr><td><b>Saldo final:</b></td><td style="padding-left:15px;color:#2e7d32">
            <b>R$ {sf:,.2f}</b></td></tr>
        <tr><td><b>Tempo total:</b></td><td style="padding-left:15px">
            {anos} anos e {meses_resto} meses ({res["meses_total"]} meses)</td></tr>
        <tr><td><b>Total investido:</b></td><td style="padding-left:15px">
            R$ {ti:,.2f}</td></tr>
        <tr><td><b>Total em juros:</b></td><td style="padding-left:15px;color:#1565C0">
            R$ {tj:,.2f}</td></tr>
        <tr><td><b>Juros / Total:</b></td><td style="padding-left:15px">
            {tj/sf*100:.1f}% do montante final</td></tr>
        <tr><td><b>Estrategia:</b></td><td style="padding-left:15px">
            {"Juros primeiro" if est=="aporte_depois" else "Aporte primeiro"}</td></tr>
      </table>
    </div>
    '''))

    # ======== GRAFICO 1: EVOLUCAO ========
    display(widgets.HTML('<h3>Evolucao do Patrimonio</h3>'))

    fig, ax = plt.subplots(figsize=(14, 6))
    anos_eixo = df['Ano'].values
    ax.plot(anos_eixo, df['Saldo'].values, label='Saldo acumulado',
            linewidth=2, color='#2196F3')
    ax.plot(anos_eixo, df['Aportes_Acum'].values,
            label='Total aportado', linewidth=1.5, color='#4CAF50', linestyle='--')
    ax.fill_between(anos_eixo, df['Aportes_Acum'].values, df['Saldo'].values,
                    alpha=0.15, color='#2196F3', label='Juros compostos')

    ax.axhline(y=alvo, color='#FF9800', linestyle='--', alpha=0.7,
               label=f'Meta: R$ {alvo:,.0f}')
    
    for a in [m1_m/12, (m1_m+m2_m)/12, (m1_m+m2_m+m3_m)/12]:
        if a < anos_eixo[-1]:
            ax.axvline(x=a, color='gray', linestyle=':', alpha=0.5)
            ax.text(a, ax.get_ylim()[1]*0.95, f'{a:.0f}a', ha='center',
                    fontsize=9, color='gray')

    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'R$ {x/1000:,.0f}k'))
    ax.set_xlabel('Anos')
    ax.set_ylabel('Valor')
    ax.set_title('Evolucao do Patrimonio ao Longo do Tempo')
    ax.legend(loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ======== GRAFICO 2: COMPOSICAO ========
    display(widgets.HTML('<h3>Composicao do Valor Final</h3>'))

    labels = ['Total investido', 'Rendimentos (juros)']
    valores = [ti, tj]
    cores = ['#4CAF50', '#2196F3']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.pie(valores, labels=labels, autopct='%1.1f%%', startangle=90,
            colors=cores, explode=(0, 0.05))
    ax1.set_title('Composicao do Montante Final')

    barras = ax2.bar(labels, valores, color=cores, width=0.5)
    for barra, valor in zip(barras, valores):
        ax2.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 10000,
                 f'R$ {valor:,.2f}', ha='center', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Valor (R$)')
    ax2.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'R$ {x/1000:,.0f}k'))
    ax2.set_title('Total Investido vs Rendimentos')
    plt.tight_layout()
    plt.show()

    # ======== GRAFICO 3: INFLACAO ========
    display(widgets.HTML('<h3>Inflacao — Valor Real vs Nominal</h3>'))

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(anos_eixo, df['Saldo'].values, label='Valor nominal',
            linewidth=2, color='#2196F3')

    for taxa_inf in [0.003, 0.005, 0.007]:
        inf_mensal = (1 + taxa_inf) ** (1/12) - 1
        saldo_real = df['Saldo'].values / ((1 + inf_mensal) ** df['Mes'].values)
        nome = f'Valor real ({taxa_inf*100:.1f}% a.a.)'
        ax.plot(anos_eixo, saldo_real, label=nome, linewidth=1.5, linestyle='--',
                alpha=0.7)

    ax.axhline(y=alvo, color='#FF9800', linestyle='--', alpha=0.5,
               label=f'Meta: R$ {alvo:,.0f}')
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'R$ {x/1000:,.0f}k'))
    ax.set_xlabel('Anos')
    ax.set_ylabel('Valor')
    ax.set_title('Impacto da Inflacao no Poder de Compra')
    ax.legend(loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ======== COMPARACAO DE CENARIOS ========
    display(widgets.HTML('<h3>Comparacao de Cenarios</h3>'))

    cenarios = [
        ('Cenario atual', taxa, fases, est),
        ('Otimista (+0.3% a.m.)', taxa + 0.003, fases, est),
        ('Pessimista (-0.3% a.m.)', max(0.001, taxa - 0.003), fases, est),
        ('Aporte 20% maior', taxa,
         [(m1_m, int(a1*1.2), f'R$ {int(a1*1.2)}/mes'),
          (m2_m, int(a2*1.2), f'R$ {int(a2*1.2)}/mes'),
          (m3_m, int(a3*1.2), f'R$ {int(a3*1.2)}/mes'),
          (None, int(a4*1.2), f'R$ {int(a4*1.2)}/mes')], est),
    ]
    if mi > 0 and si > 0:
        cenarios = [(n, t, f, e) for n, t, f, e in cenarios]

    dados_cen = []
    for nome_cen, t_cen, f_cen, e_cen in cenarios:
        if mi > 0 and si > 0:
            r = continuar_de(t_cen, alvo, f_cen, mi, si, e_cen)
        else:
            r = simular(t_cen, alvo, f_cen, estrategia=e_cen)
        dados_cen.append({
            'Cenario': nome_cen,
            'Tempo (anos)': f'{r["anos"]}a {r["meses_resto"]}m',
            'Saldo final': f'R$ {r["saldo_final"]:,.2f}',
            'Total investido': f'R$ {r["total_investido"]:,.2f}',
            'Juros': f'R$ {r["total_juros"]:,.2f}',
            'Juros %': f'{r["total_juros"]/r["saldo_final"]*100:.1f}%'
        })

    df_cen = pd.DataFrame(dados_cen)
    display(df_cen)

    # ======== HEATMAP DE SENSIBILIDADE ========
    display(widgets.HTML('<h3>Mapa de Calor — Sensibilidade (Taxa vs Aporte)</h3>'))
    display(widgets.HTML(
        '<p>Tempo em <b>anos</b> para atingir a meta variando taxa mensal '
        'e aporte unico (aporte fixo, sem fases).</p>'))

    taxas_h = np.arange(0.003, 0.020, 0.001)
    aportes_h = np.arange(200, 3000, 200)
    matrix = np.full((len(aportes_h), len(taxas_h)), np.nan)

    for i, ap_h in enumerate(aportes_h):
        for j, tx_h in enumerate(taxas_h):
            r_h = simular(tx_h, alvo, [(None, ap_h, 'Unico')],
                          estrategia=est)
            if r_h['meses_total'] > 0 and r_h['meses_total'] <= 1200:
                matrix[i, j] = r_h['meses_total'] / 12

    fig, ax = plt.subplots(figsize=(14, 7))
    cmap = plt.cm.viridis
    cmap.set_bad(color='gray', alpha=0.3)
    im = ax.imshow(matrix, aspect='auto', origin='lower', cmap=cmap,
                   interpolation='nearest')

    ax.set_xticks(range(len(taxas_h)))
    ax.set_xticklabels([f'{t*100:.1f}%' for t in taxas_h], rotation=45, ha='right')
    ax.set_yticks(range(len(aportes_h)))
    ax.set_yticklabels([f'R$ {a}' for a in aportes_h])
    ax.set_xlabel('Taxa mensal')
    ax.set_ylabel('Aporte mensal (R$)')
    ax.set_title('Anos para atingir a meta')

    cbar = plt.colorbar(im, ax=ax, fraction=0.046)
    cbar.set_label('Anos')

    for i in range(len(aportes_h)):
        for j in range(len(taxas_h)):
            v = matrix[i, j]
            if not np.isnan(v):
                color = 'white' if v > 15 else 'black'
                ax.text(j, i, f'{v:.0f}', ha='center', va='center',
                        fontsize=7, color=color, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # ======== MARCOS DA META ========
    display(widgets.HTML('<h3>Marcos da Meta</h3>'))

    if marcos:
        dados_marco = []
        for pct in [0.25, 0.50, 0.75, 1.00]:
            if pct in marcos:
                m = marcos[pct]
                dados_marco.append({
                    'Marco': f'{m["pct"]:.0f}%',
                    'Mes': m['mes'],
                    'Ano': f'{m["ano"]:.1f}',
                    'Saldo': f'R$ {m["saldo"]:,.2f}'
                })
        df_marcos = pd.DataFrame(dados_marco)

        fig, ax = plt.subplots(figsize=(12, 3))
        ax.axis('off')
        tbl = ax.table(cellText=df_marcos.values,
                       colLabels=df_marcos.columns,
                       cellLoc='center', loc='center',
                       colWidths=[0.1, 0.1, 0.1, 0.25])
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(12)
        tbl.scale(1, 2)
        for key, cell in tbl.get_celld().items():
            if key[0] == 0:
                cell.set_facecolor('#1565C0')
                cell.set_text_props(color='white', fontweight='bold')
            elif key[1] == 3:
                cell.set_facecolor('#E8F5E9')
        ax.set_title('Marcos da Meta', fontsize=14, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()
    else:
        display(widgets.HTML('<p>Nenhum marco atingido.</p>'))

    # ======== PLANILHA MES A MES ========
    display(widgets.HTML('<h3>Planilha Mes a Mes</h3>'))
    display(widgets.HTML(
        '<p>Mostrando as primeiras e ultimas linhas da simulacao completa.</p>'))

    df_display = df[['Mes', 'Ano', 'Fase', 'Aporte', 'Rendimento',
                     'Saldo', '%Meta']].copy()
    df_display.columns = ['Mes', 'Ano', 'Fase', 'Aporte (R$)', 'Rendimento (R$)',
                          'Saldo (R$)', '% da Meta']

    with pd.option_context('display.max_rows', 60,
                           'display.float_format', '{:,.2f}'.format):
        if len(df_display) <= 60:
            display(df_display)
        else:
            display(widgets.HTML('<p><b>Primeiras 30 linhas:</b></p>'))
            display(df_display.head(30))
            display(widgets.HTML('<p><b>Ultimas 30 linhas:</b></p>'))
            display(df_display.tail(30))

    display(widgets.HTML(f'''
    <p><b>Total de linhas:</b> {len(df)} &nbsp;|&nbsp;
       <b>Total investido:</b> R$ {ti:,.2f} &nbsp;|&nbsp;
       <b>Saldo final:</b> R$ {sf:,.2f}</p>
    '''))

    # ======== EXPORT ========
    display(widgets.HTML('<h3>Exportar Dados</h3>'))
    display(widgets.HTML('''
    <p>Descomente as linhas abaixo na celula de exportacao para gerar CSV ou Excel.</p>
    <pre>
    # df.to_csv('simulacao_mes_a_mes.csv', index=False, decimal=',', sep=';')
    # df.to_excel('simulacao_mes_a_mes.xlsx', index=False)
    </pre>
    '''))


out = widgets.interactive_output(
    atualizar, {
        'taxa_pct': taxa_slider,
        'alvo': alvo_input,
        'a1': ap1, 'a2': ap2, 'a3': ap3, 'a4': ap4,
        'm1_m': m1, 'm2_m': m2, 'm3_m': m3,
        'mi': mes_ini, 'si': saldo_ini,
        'est': estrategia_dd
    }
)

interface = widgets.VBox([
    widgets.HTML('<h2>Parametros da Simulacao</h2>'),
    config_box,
    widgets.HTML('<hr><h3>Aportes por Fase</h3>'),
    widgets.HTML('<p>Ajuste os valores mensais e a duracao de cada fase.</p>'),
    widgets.HBox([aporte_box, fases_box]),
    widgets.HTML('<hr>'),
    out
])

display(interface)


In [ ]:
# Exportar para CSV
# df.to_csv('simulacao_mes_a_mes.csv', index=False, decimal=',', sep=';')

# Exportar para Excel (instale openpyxl: pip install openpyxl)
# df.to_excel('simulacao_mes_a_mes.xlsx', index=False)

print("Pronto! Descomente as linhas acima para exportar.")
